# Smart metering: DLMS/COSEM dan M-Bus

Meter listrik berbicara DLMS/COSEM (IEC 62056); meter panas dan air di gedung tergantung pada M-Bus berkabel. Di sini meter rumah tangga simulasi dengan panel surya atap dan segmen M-Bus simulasi menggantikan perangkat kerasnya.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.9.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.9.0-preview.1"

## Kode OBIS dan data COSEM
Setiap nilai di meter punya nama logis enam grup; nilainya dikirim sebagai A-XDR.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Dlms;

var energyCode = ObisCode.Parse("1-0:1.8.0*255");
Console.WriteLine($"{energyCode} = {energyCode.Description}");
var scalerUnit = CosemData.Structure(CosemData.Int8(-1), CosemData.Enum(CosemUnit.Volt));
Console.WriteLine($"{scalerUnit} → {Convert.ToHexString(scalerUnit.Encode())}");

## Membaca meter
Public client (SAP 16) berasosiasi tanpa password: boleh membaca, tidak pernah menulis. Register membawa scaler dan unit; profil dibaca dengan selective access berdasarkan tanggal.

In [ ]:
using IoTCom.Net.Transports;

var meterLink = new InMemoryTransportListener("meter");
await using var meterServer = DlmsServer.Create(o => o.ListenInMemory(meterLink));
await using var householdMeter = new DlmsMeterSimulator(meterServer);
await meterServer.StartAsync();

await using var reader = DlmsClient.Create(o => o.UseInMemory(meterLink));
await reader.ConnectAsync();
foreach (var code in new[] { "1.0.1.8.0.255", "1.0.2.8.0.255", "1.0.32.7.0.255", "1.0.14.7.0.255" })
    Console.WriteLine($"{ObisCode.Parse(code).Description,-34} {await reader.ReadRegisterAsync(ObisCode.Parse(code))}");
var meterNow = await reader.ReadClockAsync();
var lastHour = await reader.ReadProfileAsync(ObisCode.Parse("1.0.99.1.0.255"), meterNow.AddHours(-1), meterNow);
Console.WriteLine($"{lastHour.Rows.Count} load-profile rows in the last hour");

## Hak tulis harus diperoleh
Public client tidak bisa membuka relay; management client dengan password bisa.

In [ ]:
// Even with the client-side read-only guard off, the meter refuses the public client.
await using var curious = DlmsClient.Create(o => { o.UseInMemory(meterLink); o.ReadOnly = false; });
await curious.ConnectAsync();
try { await curious.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 1); }
catch (DlmsException e) { Console.WriteLine($"public client: {e.Message}"); }
await using var engineer = DlmsClient.Create(o => { o.UseInMemory(meterLink).WithPassword("12345678"); o.ReadOnly = false; });
await engineer.ConnectAsync();
await engineer.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 1);
Console.WriteLine($"relay connected: {householdMeter.Relay.Connected}");
await engineer.ActionAsync(CosemClass.DisconnectControl, ObisCode.Parse("0.0.96.3.10.255"), 2);

## M-Bus: memindai segmen
Master melakukan ping ke alamat primer, lalu membaca record data variabel setiap slave.

In [ ]:
using IoTCom.Net.Protocols.MBus;

var busLink = new InMemoryTransportListener("mbus");
await using var segment = MBusSlaveSimulator.Create(o => o.ListenInMemory(busLink)).AddDefaultDevices();
await segment.StartAsync();
await using var master = MBusMaster.Create(o => { o.UseInMemory(busLink); o.ResponseTimeout = TimeSpan.FromMilliseconds(100); });
await master.ConnectAsync();
foreach (var address in await master.ScanAsync(0, 5))
{
    var telegram = await master.ReadAsync(address);
    Console.WriteLine($"[{address}] {telegram.SecondaryAddress} {telegram.MediumName}");
    foreach (var record in telegram.Records.Take(3)) Console.WriteLine($"    {record}");
}

## Lebih lanjut
Demo Galeri *Pembacaan smart meter* menggambar hari rumah tangga dari load profile dan meter M-Bus gedung. `iotcom dlms read --sim` dan `iotcom mbus scan --sim` melakukan hal yang sama di terminal. Lihat `docs/id/protocols/dlms.md` dan `docs/id/protocols/mbus.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*